# Metryki

## 1. Konfiguracja

In [34]:
ZESTAW = "symulacja"

USTAWIENIA = {
    "1sesja":      {"bands": "125,250,500,1000,2000,4000,8000,16000", "cisza": "segment"},
    "2sesja_2spk": {"bands": "2000,4000,8000,16000", "cisza": "okno"},
    "2sesja_4spk": {"bands": "2000,4000,8000,16000", "cisza": "okno"},
    "symulacja":   {"bands": "125,250,500,1000,2000,4000,8000,16000", "cisza": "okno"},
}

DANE = f"dane/{ZESTAW}"
DECODED_DIR = f"wyniki/{ZESTAW}"
MANIFEST = f"{DANE}/manifest.json"
BANDS_TEXT = USTAWIENIA[ZESTAW]["bands"]
TRYB_CISZY = USTAWIENIA[ZESTAW]["cisza"]
NOISE_SEGMENT = "cisza_koncowa_noise_floor"
NOISE_WINDOW_S = 2.0
MAX_HZ = 20000.0
OUT_CSV = f"{DECODED_DIR}/tabela.csv"

{"zestaw": ZESTAW, "tryb_ciszy": TRYB_CISZY, "pasma": BANDS_TEXT}

{'zestaw': 'symulacja',
 'tryb_ciszy': 'okno',
 'pasma': '125,250,500,1000,2000,4000,8000,16000'}

In [35]:
import csv
import json
import math
import os
import re
from copy import deepcopy
from dataclasses import dataclass
import numpy as np
import pandas as pd
import soundfile as sf
import decode_bundle

EPS = 1e-30
MAX_LAG_MS = 3.0
FULL_LOW_HZ = 1500.0

## 2. Pasma

In [36]:
@dataclass(frozen=True)
class Band:
    label: str
    center_hz: float | None
    low_hz: float | None
    high_hz: float | None


def parse_bands(text, fs, max_hz):
    bands = [Band("full", None, None, None)]
    upper = fs / 2.0 if max_hz is None else min(fs / 2.0, float(max_hz))
    for part in text.split(","):
        if not part.strip():
            continue
        c = float(part)
        if c / math.sqrt(2.0) >= upper:
            continue
        bands.append(Band(str(int(c)), c, c / math.sqrt(2.0),
                          min(c * math.sqrt(2.0), upper)))
    return bands


def band_limited(ir, fs, band):
    x = np.asarray(ir, dtype=float)
    if band.center_hz is None or x.size == 0:
        return x.copy()
    spec = np.fft.rfft(x)
    freqs = np.fft.rfftfreq(x.size, d=1.0 / fs)
    spec[~((freqs >= band.low_hz) & (freqs <= band.high_hz))] = 0.0
    return np.fft.irfft(spec, n=x.size)


def band_edges(band, max_hz):
    if band.center_hz is None:
        return FULL_LOW_HZ, (max_hz or 20000.0)
    return float(band.low_hz), float(band.high_hz)


with open(MANIFEST, encoding="utf-8") as fh:
    meta = json.load(fh)

FS = float(meta["fs"])
N_SPK = meta["speakers"]
BANDS = parse_bands(BANDS_TEXT, FS, MAX_HZ)

pd.DataFrame([{"pasmo": b.label, "od": b.low_hz, "do": b.high_hz} for b in BANDS])

,pasmo,od,do
0,full,NaN,NaN
1,125,88.388348,176.776695
2,250,176.776695,353.553391
3,500,353.553391,707.106781
4,1000,707.106781,1414.213562
5,2000,1414.213562,2828.427125
6,4000,2828.427125,5656.854249
7,8000,5656.854249,11313.708499
8,16000,11313.708499,20000.000000


## 3. Wczytanie IR

In [37]:
DECODED_RE = re.compile(r"^decoded_(?P<method>.+)_spk(?P<speaker>\d+)\.npy$")

decoded = {}
for filename in sorted(os.listdir(DECODED_DIR)):
    m = DECODED_RE.match(filename)
    if m:
        ir = np.asarray(np.load(os.path.join(DECODED_DIR, filename)), dtype=float)
        decoded.setdefault(m.group("method"), {})[int(m.group("speaker"))] = ir

IR_LEN = max(ir.size for spk in decoded.values() for ir in spk.values())
{"metody": sorted(decoded), "ir_len": IR_LEN, "ms": round(IR_LEN / FS * 1000, 1)}

{'metody': ['golay_hadamard',
  'gold',
  'mls_hadamard',
  'mls_reciprocal',
  'solo_golay',
  'solo_mls'],
 'ir_len': 4096,
 'ms': 85.3}

## 4. Cisza

In [38]:
def quietest_window(y, n):
    if y.size < n:
        return y.copy(), 0
    cum = np.concatenate(([0.0], np.cumsum(y.astype(np.float64) ** 2)))
    start = int(np.argmin(cum[n:] - cum[:-n]))
    return y[start:start + n].copy(), start


def wczytaj_mono(path):
    data, fs = sf.read(path, always_2d=False)
    y = np.asarray(data, dtype=np.float64)
    return (y[:, 0] if y.ndim > 1 else y), fs


if TRYB_CISZY == "segment":
    y_noise, fs_noise = wczytaj_mono(f"{DANE}/cisza.wav")
    with open(f"{DANE}/calibration_manifest.json", encoding="utf-8") as fh:
        cal = json.load(fh)
    seg = next(s for s in cal["segments"] if s["name"] == NOISE_SEGMENT)
    noise = y_noise[int(seg["start"]):int(seg["end"])]
    zrodlo = f"segment {NOISE_SEGMENT}"
else:
    plik = f"{DANE}/cisza.wav" if os.path.exists(f"{DANE}/cisza.wav")         else f"{DANE}/nagranie.wav"
    y_noise, fs_noise = wczytaj_mono(plik)
    noise, start = quietest_window(y_noise, int(round(NOISE_WINDOW_S * fs_noise)))
    zrodlo = f"najcichsze {NOISE_WINDOW_S} s z {os.path.basename(plik)} (od {start/fs_noise:.2f} s)"

noise = noise - float(np.mean(noise))
{"zrodlo": zrodlo, "probek": noise.size}

{'zrodlo': 'najcichsze 2.0 s z nagranie.wav (od 3.25 s)', 'probek': 96000}

In [39]:
WYMAGANY_SEGMENT = {
    "solo_golay": "solo_golay_A", "solo_mls": "solo_mls",
    "golay_hadamard": "golay_hadamard_A", "mls_hadamard": "mls_hadamard",
    "mls_reciprocal": "mls_reciprocal", "gold": "gold",
}

KINDS = set(WYMAGANY_SEGMENT.values()) | {"solo_golay_B", "golay_hadamard_B",
                                          "mls_reciprocal_pair"}
golay_len = int(meta["golay_L"])
ends = [int(s["end"]) for s in meta["segments"]]
ends += [int(s["start"]) + golay_len + IR_LEN
         for s in meta["segments"] if s.get("kind") in KINDS]
n_needed = max(ends)

tiled = np.tile(noise, int(math.ceil(n_needed / noise.size)))[:n_needed].copy()
obecne = {s["kind"] for s in meta["segments"]}

noise_decoded = {}
for method in decode_bundle.METHODS:
    if WYMAGANY_SEGMENT[method.key] not in obecne:
        continue
    irs = method.decode(tiled, deepcopy(meta), IR_LEN)
    noise_decoded[method.key] = {i + 1: ir for i, ir in enumerate(irs)}

sorted(noise_decoded)

['golay_hadamard',
 'gold',
 'mls_hadamard',
 'mls_reciprocal',
 'solo_golay',
 'solo_mls']

## 5. SNR i C80

In [40]:
def db10(v):
    return 10.0 * math.log10(v) if v > 0.0 else float("-inf")


def signal_power(ir, fs, tail_fraction=0.10):
    n = min(max(8, int(round(ir.size * tail_fraction))), max(1, ir.size // 2))
    sig = ir[:-n] if 0 < n < ir.size else ir
    return float(np.mean(sig * sig))


def c80_db(ir, fs, c80_ms=80.0):
    split = int(round(c80_ms * fs / 1000.0))
    if split <= 0 or split >= ir.size:
        return float("nan")
    return db10(float(np.sum(ir[:split] ** 2)) / max(float(np.sum(ir[split:] ** 2)), EPS))


values = {}
for method in sorted(decoded):
    for speaker in sorted(decoded[method]):
        full_ir = decoded[method][speaker]
        noise_ir = noise_decoded.get(method, {}).get(speaker)
        for band in BANDS:
            ir_b = band_limited(full_ir, FS, band)
            snr = None
            if noise_ir is not None:
                nb_ = band_limited(np.asarray(noise_ir)[:full_ir.size], FS, band)
                snr = db10(signal_power(ir_b, FS) / max(float(np.mean(nb_ * nb_)), EPS))
            values[(method, speaker, band.label)] = {"snr": snr, "c80": c80_db(ir_b, FS)}

pd.DataFrame([{"metoda": m, "spk": s, "pasmo": b, **v}
              for (m, s, b), v in values.items()]).pivot_table(
    index=["metoda", "spk"], columns="pasmo", values="snr").round(2)

pasmo                1000    125  16000   2000    250   4000    500   8000  \
metoda         spk                                                           
golay_hadamard 1    82.64  78.18  85.21  86.16  80.05  90.65  80.48  87.76   
               2    83.09  78.09  68.82  82.84  76.07  86.81  79.12  78.93   
gold           1    73.87  69.97  76.21  77.20  71.46  82.09  72.69  78.71   
               2    73.81  67.41  64.76  73.73  68.22  78.62  70.33  72.06   
mls_hadamard   1    79.57  73.85  82.11  83.28  75.31  87.99  78.68  84.97   
               2    78.86  71.61  66.00  80.05  74.17  84.22  74.66  75.87   
mls_reciprocal 1    77.72  71.03  79.24  80.41  77.00  84.73  74.87  82.09   
               2    77.88  69.45  66.49  77.36  72.87  81.04  72.97  73.62   
solo_golay     1    80.08  77.20  82.42  82.66  75.46  87.87  77.47  84.71   
               2    80.10  73.54  65.71  79.89  75.94  83.44  75.17  76.40   
solo_mls       1    75.43  73.16  79.26  80.42  73.27  85.15  75.60  81.87   
               2    76.89  69.24  63.03  76.26  72.17  80.76  72.72  73.23   

pasmo                full  
metoda         spk         
golay_hadamard 1    86.46  
               2    79.99  
gold           1    77.63  
               2    71.97  
mls_hadamard   1    83.56  
               2    76.97  
mls_reciprocal 1    80.70  
               2    74.55  
solo_golay     1    83.62  
               2    76.95  
solo_mls       1    80.59  
               2    74.00

## 6. Metryki różnicowe

In [41]:
def shift_signal(x, s):
    if s == 0:
        return x.copy()
    y = np.zeros_like(x)
    if s > 0:
        y[s:] = x[:-s]
    else:
        y[:s] = x[-s:]
    return y


def align_to_reference(reference, test, fs):
    n = min(reference.size, test.size)
    ref, tst = reference[:n], test[:n]
    max_lag = min(int(round(MAX_LAG_MS * fs / 1000.0)), max(n - 1, 0))
    ref_e = float(np.dot(ref, ref))
    best_lag, best = 0, -float("inf")
    for lag in range(-max_lag, max_lag + 1):
        sh = shift_signal(tst, lag)
        den = math.sqrt(max(ref_e * float(np.dot(sh, sh)), EPS))
        sc = float(np.dot(ref, sh)) / den
        if sc > best:
            best, best_lag = sc, lag
    return shift_signal(test, best_lag)


def smooth_oct(f, y, fraction=3, fmin=20.0):
    f, y = np.asarray(f, dtype=float), np.asarray(y, dtype=float)
    out = y.copy()
    i0 = int(np.argmax(f >= fmin))
    lf = np.log2(f[i0:])
    hb = 1.0 / (2.0 * fraction)
    for k in range(lf.size):
        lo = int(np.searchsorted(lf, lf[k] - hb, side="left"))
        hi = int(np.searchsorted(lf, lf[k] + hb, side="right"))
        if hi > lo:
            out[i0 + k] = float(np.mean(y[i0 + lo:i0 + hi]))
    return out


def complex_band_correlation(h_ref, h_test, freqs, lo, hi):
    mask = (freqs >= lo) & (freqs <= hi)
    if not np.any(mask):
        return float("nan")
    a, b = h_ref[mask], h_test[mask]
    den = math.sqrt(max(float(np.sum(np.abs(a) ** 2)) * float(np.sum(np.abs(b) ** 2)), EPS))
    return abs(complex(np.sum(a * np.conj(b)) / den))


def pair_metrics(reference, test, fs, bands, max_hz):
    n = min(reference.size, test.size)
    reference, test = reference[:n], test[:n]
    aligned = align_to_reference(reference, test, fs)[:n]

    nfft = int(2 ** math.ceil(math.log2(max(n, 2) * 4)))
    h_ref = np.fft.rfft(reference, n=nfft)
    h_test = np.fft.rfft(aligned, n=nfft)
    freqs = np.fft.rfftfreq(nfft, d=1.0 / fs)

    win = np.hanning(n)
    ref_mag = np.abs(np.fft.rfft(reference * win))
    test_mag = np.abs(np.fft.rfft(test * win))
    freqs_h = np.fft.rfftfreq(n, d=1.0 / fs)
    dmag = (smooth_oct(freqs_h, 20.0 * np.log10(np.maximum(test_mag, 1e-12)))
            - smooth_oct(freqs_h, 20.0 * np.log10(np.maximum(ref_mag, 1e-12))))

    out = {}
    for band in bands:
        ref_b = band_limited(reference, fs, band)
        err = band_limited(aligned, fs, band) - ref_b
        rel = (math.sqrt(float(np.mean(err * err)))
               / max(math.sqrt(float(np.mean(ref_b * ref_b))), EPS))
        lo, hi = band_edges(band, max_hz)
        seg = dmag[(freqs_h >= lo) & (freqs_h <= hi)]
        out[band.label] = {
            "rmse": 20.0 * math.log10(max(rel, EPS)),
            "corr": complex_band_correlation(h_ref, h_test, freqs, lo, hi),
            "dmag": float(seg[int(np.argmax(np.abs(seg)))]) if seg.size else float("nan"),
        }
    return out


demo = pair_metrics(decoded["solo_golay"][1], decoded["golay_hadamard"][1],
                    FS, BANDS, MAX_HZ)
pd.DataFrame(demo).T.round(3)

,rmse,corr,dmag
full,-81.365,1.0,-1.953
125,-74.218,1.0,-0.346
250,-74.368,1.0,0.145
500,-76.896,1.0,-0.175
1000,-78.174,1.0,0.031
2000,-81.395,1.0,0.017
4000,-85.288,1.0,-0.059
8000,-82.701,1.0,-0.191
16000,-80.320,1.0,-1.953


## 7. Tabela

In [42]:
METHOD_LABELS = {
    "golay_hadamard": "Golay+Had.", "solo_golay": "Solo Golay",
    "mls_hadamard": "MLS+Had.", "solo_mls": "Solo MLS",
    "mls_reciprocal": "MLS recip.", "gold": "Gold",
}
COMPARISON_SPECS = [
    ("golay_hadamard", "solo_golay", "GH-Solo"),
    ("mls_hadamard", "solo_mls", "MLSH-Solo"),
    ("mls_reciprocal", "gold", "MLSrec-Gold"),
]
PAIR_ROWS = [("|rc|", "corr", 3), ("dMag [dB]", "dmag", 2), ("RMSE [dB]", "rmse", 2)]

labels = [b.label for b in BANDS]
header = ["Metoda", "Gl.", "Parametr"] + \
         ["pelne" if b == "full" else f"{b} Hz" for b in labels]


def fmt(v, dec):
    if v is None:
        return "-"
    v = float(v)
    return "-" if math.isnan(v) or math.isinf(v) else f"{v:.{dec}f}"


def seria(method, speaker, key):
    return [values.get((method, speaker, b), {}).get(key) for b in labels]


def subtract(a, b):
    return [None if x is None or y is None else x - y for x, y in zip(a, b)]


rows = []
for key in METHOD_LABELS:
    if key not in decoded:
        continue
    for spk in sorted(decoded[key]):
        rows.append([METHOD_LABELS[key], str(spk), "SNR [dB]"] +
                    [fmt(v, 2) for v in seria(key, spk, "snr")])
        rows.append(["", "", "C80 [dB]"] + [fmt(v, 2) for v in seria(key, spk, "c80")])

for test_m, ref_m, label in COMPARISON_SPECS:
    if test_m not in decoded or ref_m not in decoded:
        continue
    for spk in sorted(set(decoded[test_m]) & set(decoded[ref_m])):
        pm = pair_metrics(decoded[ref_m][spk], decoded[test_m][spk], FS, BANDS, MAX_HZ)
        rows.append([f"d {label}", f"{spk}-{spk}", "dSNR [dB]"] +
                    [fmt(v, 2) for v in subtract(seria(test_m, spk, "snr"),
                                                 seria(ref_m, spk, "snr"))])
        rows.append(["", "", "dC80 [dB]"] +
                    [fmt(v, 2) for v in subtract(seria(test_m, spk, "c80"),
                                                 seria(ref_m, spk, "c80"))])
        for param, k, dec in PAIR_ROWS:
            rows.append(["", "", param] + [fmt(pm[b][k], dec) for b in labels])

pd.DataFrame(rows, columns=header)

,Metoda,Gl.,Parametr,pelne,125 Hz,250 Hz,500 Hz,1000 Hz,2000 Hz,4000 Hz,8000 Hz,16000 Hz
0,Golay+Had.,1,SNR [dB],86.46,78.18,80.05,80.48,82.64,86.16,90.65,87.76,85.21
1,,,C80 [dB],97.32,3.75,4.32,4.42,9.12,12.13,19.26,16.40,16.30
2,Golay+Had.,2,SNR [dB],79.99,78.09,76.07,79.12,83.09,82.84,86.81,78.93,68.82
3,,,C80 [dB],91.83,4.05,4.07,3.93,11.37,12.84,19.96,18.56,15.50
4,Solo Golay,1,SNR [dB],83.62,77.20,75.46,77.47,80.08,82.66,87.87,84.71,82.42
5,,,C80 [dB],94.82,3.75,4.32,4.42,9.12,12.13,19.26,16.40,16.30
6,Solo Golay,2,SNR [dB],76.95,73.54,75.94,75.17,80.10,79.89,83.44,76.40,65.71
7,,,C80 [dB],88.20,4.05,4.07,3.93,11.37,12.84,19.96,18.56,15.49
8,MLS+Had.,1,SNR [dB],83.56,73.85,75.31,78.68,79.57,83.28,87.99,84.97,82.11
9,,,C80 [dB],43.19,3.82,4.37,4.43,9.14,12.20,19.21,16.39,16.25


## 8. Walidacja (tylko symulacja)

In [43]:
TRUE_DIR = f"{DANE}/true_irs"

if os.path.isdir(TRUE_DIR):
    true_irs = [np.load(f"{TRUE_DIR}/true_h{j+1}.npy") for j in range(N_SPK)]

    def dopasuj(est, ref):
        n = min(est.size, ref.size)
        e, r = est[:n], ref[:n]
        return {"korelacja": float(np.dot(e, r) /
                                   max(np.sqrt(np.dot(e, e) * np.dot(r, r)), 1e-30))}

    walidacja = pd.DataFrame(
        [{"metoda": METHOD_LABELS.get(k, k), "spk": s, **dopasuj(decoded[k][s], true_irs[s - 1])}
         for k in sorted(decoded) for s in sorted(decoded[k])]).round(4)
else:
    walidacja = "brak true_irs w tym zestawie"

walidacja

,metoda,spk,korelacja
0,Golay+Had.,1,1.0000
1,Golay+Had.,2,1.0000
2,Gold,1,0.9997
3,Gold,2,0.9894
4,MLS+Had.,1,1.0000
5,MLS+Had.,2,1.0000
6,MLS recip.,1,0.9998
7,MLS recip.,2,0.9969
8,Solo Golay,1,1.0000
9,Solo Golay,2,1.0000


## 9. Zapis

In [44]:
with open(OUT_CSV, "w", newline="", encoding="utf-8-sig") as fh:
    w = csv.writer(fh, delimiter=";")
    w.writerow(header)
    for r in rows:
        w.writerow([c.replace(".", ",") if c not in ("-",) and
                    c.replace("-", "").replace(".", "").isdigit() else c for c in r])

{"plik": os.path.abspath(OUT_CSV), "wierszy": len(rows)}

{'plik': 'c:\\Users\\kamil\\Desktop\\pipeline_2sesja\\wysylka\\colab\\wyniki\\symulacja\\tabela.csv',
 'wierszy': 54}